# Bank Marketing Campaign — XGBoost Modelling

## Setup & Data Loading 

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
import time

from sklearn.linear_model    import LogisticRegression
from sklearn.tree            import DecisionTreeClassifier
from sklearn.ensemble        import RandomForestClassifier
from xgboost                 import XGBClassifier

from sklearn.model_selection import (
    StratifiedKFold, cross_val_score,
    GridSearchCV, RandomizedSearchCV
)
from sklearn.metrics import (
    f1_score, recall_score, precision_score,
    roc_auc_score, accuracy_score,
    confusion_matrix, classification_report,
    RocCurveDisplay, PrecisionRecallDisplay
)
import shap

plt.rcParams['figure.dpi'] = 130
plt.rcParams['font.family'] = 'DejaVu Sans'

NAVY  = '#1F3864'
BLUE  = '#2E75B6'
TEAL  = '#1ABC9C'
RED   = '#E74C3C'
AMBER = '#F39C12'
GRAY  = '#BDC3C7'

In [4]:
# Load preprocessed data 
TRAIN_URL = "https://raw.githubusercontent.com/kup-kup/telemarketing-prediction/refs/heads/main/processed-data/train.csv"
TEST_URL  = "https://raw.githubusercontent.com/kup-kup/telemarketing-prediction/refs/heads/main/processed-data/test.csv"

df_train = pd.read_csv(TRAIN_URL)
df_test  = pd.read_csv(TEST_URL)

TARGET = 'y'

X_train = df_train.drop(columns=[TARGET])
y_train = df_train[TARGET].astype(int)   # True/False → 1/0

X_test  = df_test.drop(columns=[TARGET])
y_test  = df_test[TARGET].astype(int)

FEATURE_NAMES    = X_train.columns.tolist()
X_test = X_test[FEATURE_NAMES]
scale_pos_weight = round(y_train.value_counts()[0] / y_train.value_counts()[1], 2)

# Cross-validation 5-fold
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print(f'Train : {X_train.shape[0]:,} rows × {X_train.shape[1]} features')
print(f'Test  : {X_test.shape[0]:,} rows  × {X_test.shape[1]} features')
print(f'Train — No: {(y_train==0).sum():,}  Yes: {y_train.sum():,}')
print(f'Test  — No: {(y_test==0).sum():,}   Yes: {y_test.sum():,}')
print(f'Imbalance ratio   : {scale_pos_weight}:1')
print(f'scale_pos_weight  : {scale_pos_weight}')
print(f'Missing values    : {X_train.isnull().sum().sum()}')

Train : 32,950 rows × 37 features
Test  : 8,238 rows  × 37 features
Train — No: 29,238  Yes: 3,712
Test  — No: 7,310   Yes: 928
Imbalance ratio   : 7.88:1
scale_pos_weight  : 7.88
Missing values    : 0


In [5]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32950 entries, 0 to 32949
Data columns (total 38 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   age                32950 non-null  float64
 1   education          32950 non-null  float64
 2   campaign           32950 non-null  float64
 3   previous           32950 non-null  float64
 4   cons.conf.idx      32950 non-null  float64
 5   y                  32950 non-null  bool   
 6   marital_married    32950 non-null  bool   
 7   marital_single     32950 non-null  bool   
 8   job_blue-collar    32950 non-null  bool   
 9   job_entrepreneur   32950 non-null  bool   
 10  job_housemaid      32950 non-null  bool   
 11  job_management     32950 non-null  bool   
 12  job_retired        32950 non-null  bool   
 13  job_self-employed  32950 non-null  bool   
 14  job_services       32950 non-null  bool   
 15  job_student        32950 non-null  bool   
 16  job_technician     329

## Baseline XGBoost
Train XGBoost with default parameters + `scale_pos_weight` only 

In [6]:
v1_model = XGBClassifier(
    scale_pos_weight  = scale_pos_weight,
    random_state      = 42,
    eval_metric       = 'logloss',
    use_label_encoder = False
)

# CV 5 folds
v1_cv = cross_val_score(v1_model, X_train, y_train, cv=cv, scoring='f1', n_jobs=-1)

print(f'\n5-Fold CV F1 scores:')
for i, s in enumerate(v1_cv, 1):
    print(f'  Fold {i}: {s:.4f}')
print(f'  Mean : {v1_cv.mean():.4f}')

# train
v1_model.fit(X_train, y_train)

# test
v1_pred       = v1_model.predict(X_test)
v1_pred_proba = v1_model.predict_proba(X_test)[:, 1]

v1_metrics = {
    'F1 (yes)'   : f1_score(y_test, v1_pred),
    'Recall'     : recall_score(y_test, v1_pred),
    'Precision'  : precision_score(y_test, v1_pred),
    'ROC-AUC'    : roc_auc_score(y_test, v1_pred_proba),
    'Accuracy'   : accuracy_score(y_test, v1_pred),
    'CV F1 Mean' : v1_cv.mean(),
    'CV F1 Std'  : v1_cv.std()
}

print('\nBASELINE RESULTS')
for metric, value in v1_metrics.items():
    print(f'  {metric:<15}: {value:.4f}')

print('\nClassification Report:')
print(classification_report(y_test, v1_pred, target_names=['no', 'yes']))


5-Fold CV F1 scores:
  Fold 1: 0.4247
  Fold 2: 0.4296
  Fold 3: 0.4294
  Fold 4: 0.4402
  Fold 5: 0.4467
  Mean : 0.4341


c:\Users\acer\AppData\Local\Programs\Python\Python314\Lib\site-packages\xgboost\training.py:200: UserWarning: [00:00:41] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



BASELINE RESULTS
  F1 (yes)       : 0.4711
  Recall         : 0.6282
  Precision      : 0.3769
  ROC-AUC        : 0.7932
  Accuracy       : 0.8411
  CV F1 Mean     : 0.4341
  CV F1 Std      : 0.0081

Classification Report:
              precision    recall  f1-score   support

          no       0.95      0.87      0.91      7310
         yes       0.38      0.63      0.47       928

    accuracy                           0.84      8238
   macro avg       0.66      0.75      0.69      8238
weighted avg       0.88      0.84      0.86      8238



## (Hyperparameter Tuning) Phase 1: GridSearch on `scale_pos_weight` 


In [7]:
print('PHASE 1: GridSearchCV → scale_pos_weight')
grid_params = {'scale_pos_weight': [5, 7, 7.88, 10]}

grid_search = GridSearchCV(
    estimator = XGBClassifier(
        n_estimators=100, max_depth=5,
        random_state=42, eval_metric='logloss',
        use_label_encoder=False
    ),
    param_grid = grid_params,
    cv         = cv,
    scoring    = 'f1',
    verbose    = 1,
    n_jobs     = -1
)
grid_search.fit(X_train, y_train)

print(f'\nAll 4 values tested:')
print(f'{"Rank":<6} {"scale_pos_weight":<22} {"CV F1 Mean":<14} {"CV F1 Std":<12}')

for _, row in pd.DataFrame(grid_search.cv_results_).sort_values('rank_test_score').iterrows():
    spw     = row['params']['scale_pos_weight']
    print(f"  {int(row['rank_test_score']):<5} {spw:<22} "
          f"{row['mean_test_score']:.4f}{'':>8} "
          f"±{row['std_test_score']:.4f}")

BEST_SPW = grid_search.best_params_['scale_pos_weight']
print(f'\nBest scale_pos_weight : {BEST_SPW}')
print(f'Best CV F1            : {grid_search.best_score_:.4f}')

PHASE 1: GridSearchCV → scale_pos_weight
Fitting 5 folds for each of 4 candidates, totalling 20 fits


c:\Users\acer\AppData\Local\Programs\Python\Python314\Lib\site-packages\xgboost\training.py:200: UserWarning: [00:00:46] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



All 4 values tested:
Rank   scale_pos_weight       CV F1 Mean     CV F1 Std   
  1     5                      0.4752         ±0.0045
  2     7                      0.4541         ±0.0048
  3     7.88                   0.4424         ±0.0058
  4     10                     0.4086         ±0.0090

Best scale_pos_weight : 5
Best CV F1            : 0.4752


## (Hyperparameter Tuning) Phase 2: RandomizedSearch on Model Structure

In [8]:
print(f'scale_pos_weight = {BEST_SPW} (from Phase 1)')
print('Testing 80 random combinations out of 200,000+')

random_params = {
    'n_estimators'    : [200, 300, 400, 500, 700],
    'max_depth'       : [3, 4, 5, 6, 7],
    'learning_rate'   : [0.01, 0.03, 0.05, 0.08, 0.1],
    'min_child_weight': [1, 3, 5, 10],
    'subsample'       : [0.6, 0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
    'reg_alpha'       : [0, 0.01, 0.1, 0.5],
    'reg_lambda'      : [1.0, 2.0, 5.0, 10.0],
}

random_search = RandomizedSearchCV(
    estimator = XGBClassifier(
        scale_pos_weight  = BEST_SPW,
        random_state      = 42,
        eval_metric       = 'logloss',
        use_label_encoder = False
    ),
    param_distributions = random_params,
    n_iter       = 80,
    cv           = cv,
    scoring      = 'f1',
    verbose      = 1,
    random_state = 42,
    n_jobs       = -1
)
random_search.fit(X_train, y_train)

BEST_PARAMS = random_search.best_params_
print(f'\nBest params found:')
for k, v in sorted(BEST_PARAMS.items()):
    print(f'   {k:<22}: {v}')
print(f'\nBest CV F1 : {random_search.best_score_:.4f}')
print(f'   Baseline CV F1 was: {v1_metrics["CV F1 Mean"]:.4f}')
print(f'   Improvement       : +{random_search.best_score_ - v1_metrics["CV F1 Mean"]:.4f}')

# Top 5 combinations
rand_results = pd.DataFrame(random_search.cv_results_)
print(f'\nTop 5 combinations tested:')
print(f'{"Rank":<6} {"CV F1":<10} {"Std":<10} {"n_est":<7} {"depth":<7} {"lr":<7} {"subsample":<11} {"colsample"}')
print('-' * 70)
for _, row in rand_results.sort_values('rank_test_score').head(5).iterrows():
    p = row['params']
    print(f"  {int(row['rank_test_score']):<5} "
          f"{row['mean_test_score']:.4f}{'':>4} "
          f"±{row['std_test_score']:.4f}   "
          f"{p['n_estimators']:<7} {p['max_depth']:<7} "
          f"{p['learning_rate']:<7} {p['subsample']:<11} "
          f"{p['colsample_bytree']}")

scale_pos_weight = 5 (from Phase 1)
Testing 80 random combinations out of 200,000+
Fitting 5 folds for each of 80 candidates, totalling 400 fits


c:\Users\acer\AppData\Local\Programs\Python\Python314\Lib\site-packages\xgboost\training.py:200: UserWarning: [00:02:07] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



Best params found:
   colsample_bytree      : 0.6
   learning_rate         : 0.01
   max_depth             : 6
   min_child_weight      : 3
   n_estimators          : 300
   reg_alpha             : 0.01
   reg_lambda            : 5.0
   subsample             : 0.7

Best CV F1 : 0.4976
   Baseline CV F1 was: 0.4341
   Improvement       : +0.0635

Top 5 combinations tested:
Rank   CV F1      Std        n_est   depth   lr      subsample   colsample
----------------------------------------------------------------------
  1     0.4976     ±0.0069   300     6       0.01    0.7         0.6
  2     0.4973     ±0.0060   300     6       0.01    0.6         0.8
  3     0.4971     ±0.0052   300     7       0.01    0.9         0.8
  4     0.4967     ±0.0057   200     7       0.01    0.6         1.0
  5     0.4962     ±0.0031   500     7       0.01    0.9         0.8


## Version 2: Tuned XGBoost

Build the final tuned model using best parameters from Phase 1 + Phase 2. 

In [9]:
v2_model = XGBClassifier(
    scale_pos_weight  = BEST_SPW,
    n_estimators      = BEST_PARAMS['n_estimators'],
    max_depth         = BEST_PARAMS['max_depth'],
    learning_rate     = BEST_PARAMS['learning_rate'],
    min_child_weight  = BEST_PARAMS['min_child_weight'],
    subsample         = BEST_PARAMS['subsample'],
    colsample_bytree  = BEST_PARAMS['colsample_bytree'],
    reg_alpha         = BEST_PARAMS['reg_alpha'],
    reg_lambda        = BEST_PARAMS['reg_lambda'],
    random_state      = 42,
    eval_metric       = 'logloss'
)

v2_cv = cross_val_score(v2_model, X_train, y_train, cv=cv, scoring='f1', n_jobs=-1)
v2_model.fit(X_train, y_train)

v2_pred       = v2_model.predict(X_test)
v2_pred_proba = v2_model.predict_proba(X_test)[:, 1]

v2_metrics = {
    'F1 (yes)'   : f1_score(y_test, v2_pred),
    'Recall'     : recall_score(y_test, v2_pred),
    'Precision'  : precision_score(y_test, v2_pred),
    'ROC-AUC'    : roc_auc_score(y_test, v2_pred_proba),
    'Accuracy'   : accuracy_score(y_test, v2_pred),
    'CV F1 Mean' : v2_cv.mean(),
    'CV F1 Std'  : v2_cv.std()
}

print('VERSION COMPARISON: Baseline vs Tuned')
print(f'{"Metric":<16} {"V1 Baseline":>12} {"V2 Tuned":>12} {"Change":>10}')
print('-' * 55)

metrics_to_show = [
    'F1 (yes)', 'Recall', 'Precision', 'ROC-AUC', 
    'Accuracy', 'CV F1 Mean', 'CV F1 Std'
]

for metric in metrics_to_show:
    v1 = v1_metrics[metric]
    v2 = v2_metrics[metric]
    diff = v2 - v1
    diff_str = f'+{diff:.4f}' if diff > 0 else f'{diff:.4f}'
    print(f'{metric:<16} {v1:>12.4f} {v2:>12.4f} {diff_str:>10}')

print('\nClassification Report (V2 Tuned):')
print(classification_report(y_test, v2_pred, target_names=['no', 'yes']))

VERSION COMPARISON: Baseline vs Tuned
Metric            V1 Baseline     V2 Tuned     Change
-------------------------------------------------------
F1 (yes)               0.4711       0.5311    +0.0600
Recall                 0.6282       0.6207    -0.0075
Precision              0.3769       0.4641    +0.0873
ROC-AUC                0.7932       0.8167    +0.0235
Accuracy               0.8411       0.8765    +0.0354
CV F1 Mean             0.4341       0.4976    +0.0635
CV F1 Std              0.0081       0.0069    -0.0012

Classification Report (V2 Tuned):
              precision    recall  f1-score   support

          no       0.95      0.91      0.93      7310
         yes       0.46      0.62      0.53       928

    accuracy                           0.88      8238
   macro avg       0.71      0.76      0.73      8238
weighted avg       0.89      0.88      0.88      8238



## Threshold Optimisation

Default threshold = 0.5. We sweep all thresholds to find two operating points:
- **Option A**: Maximise F1
- **Option B**: Maximise F1 while keeping Recall ≥ 0.70 (business constraint)

In [14]:
xgb_proba = v2_pred_proba
thresholds  = np.arange(0.01, 0.99, 0.01)
f1_list, rec_list, prec_list = [], [], []

for t in thresholds:
    preds = (xgb_proba >= t).astype(int)
    f1_list.append(f1_score(y_test, preds, zero_division=0)) #zero_division=0 to avoid warnings when precision or recall is undefined (base = 0)
    rec_list.append(recall_score(y_test, preds, zero_division=0))
    prec_list.append(precision_score(y_test, preds, zero_division=0))

# Option A: Best F1 
best_f1_idx  = np.argmax(f1_list)
THRESHOLD_A  = thresholds[best_f1_idx]

# Option B: Best F1 with Recall ≥ 0.70 
valid_idx   = [i for i, r in enumerate(rec_list) if r >= 0.70]
best_b_idx  = valid_idx[np.argmax([f1_list[i] for i in valid_idx])]
THRESHOLD_B = thresholds[best_b_idx]

print(f'{"Threshold":<12} {"F1":>8} {"Recall":>8} {"Precision":>10}  Note')
print('-' * 55)

show_thresholds = sorted(set([0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50,
                               round(THRESHOLD_A, 2), round(THRESHOLD_B, 2)]))
for t in show_thresholds:
    idx  = np.argmin(np.abs(thresholds - t))
    note = ''
    if abs(t - THRESHOLD_A) < 0.011: note = '← Option A: Max F1'
    if abs(t - THRESHOLD_B) < 0.011 and t != THRESHOLD_A: note = '← Option B: Recall≥0.70'
    if abs(t - 0.50) < 0.011: note = '← Default'
    print(f'  {t:<10.2f} {f1_list[idx]:>8.4f} {rec_list[idx]:>8.4f} {prec_list[idx]:>10.4f}  {note}')

print(f'\n{"="*55}')
print('OPERATING POINT RECOMMENDATION')

idx_a = np.argmin(np.abs(thresholds - THRESHOLD_A))
idx_b = np.argmin(np.abs(thresholds - THRESHOLD_B))

print(f'\nOption A — Maximise F1 (threshold = {THRESHOLD_A:.2f}):')
print(f'  F1        : {f1_list[idx_a]:.4f}')
print(f'  Recall    : {rec_list[idx_a]:.4f}')
print(f'  Precision : {prec_list[idx_a]:.4f}')

print(f'\nOption B — Recall ≥ 0.70 (threshold = {THRESHOLD_B:.2f}):')
print(f'  F1        : {f1_list[idx_b]:.4f}')
print(f'  Recall    : {rec_list[idx_b]:.4f} ')
print(f'  Precision : {prec_list[idx_b]:.4f}')

pred_optA = (xgb_proba >= THRESHOLD_A).astype(int)
pred_optB = (xgb_proba >= THRESHOLD_B).astype(int)

Threshold          F1   Recall  Precision  Note
-------------------------------------------------------
  0.20         0.2543   0.9095     0.1478  
  0.25         0.3314   0.8319     0.2069  
  0.30         0.4277   0.7263     0.3031  ← Option B: Recall≥0.70
  0.31         0.4395   0.7080     0.3186  ← Option B: Recall≥0.70
  0.35         0.4614   0.6800     0.3492  
  0.40         0.4963   0.6519     0.4007  
  0.45         0.5213   0.6390     0.4402  
  0.50         0.5311   0.6207     0.4641  ← Default
  0.51         0.5331   0.6196     0.4679  ← Default

OPERATING POINT RECOMMENDATION

Option A — Maximise F1 (threshold = 0.51):
  F1        : 0.5331
  Recall    : 0.6196
  Precision : 0.4679

Option B — Recall ≥ 0.70 (threshold = 0.31):
  F1        : 0.4395
  Recall    : 0.7080 
  Precision : 0.3186
